# Stage 01 - Data Ingestion: Ingestion and Parsing of Word (DOCX) Documents
### Lecture 14: Structured Extraction from Microsoft Word Files

This notebook implements Word document parsing:
1. Ingestion using `Docx2txtLoader`.
2. Extracting headings, paragraphs, and embedded tables using `python-docx`.
3. Converting Word tables into structured markdown text representation.
4. Preserving structural hierarchy and document metadata.


In [1]:
# Step 1: Environment Setup and Target DOCX Verification
from pathlib import Path
from langchain_community.document_loaders import Docx2txtLoader
import docx

repo_root = Path.cwd()
while not (repo_root / "data").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

docx_path = repo_root / "data" / "docx" / "proposal.docx"
print(f"[CONFIG] Target DOCX: {docx_path}")
assert docx_path.exists(), f"File {docx_path} must exist."


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_23576\1672478563.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


[CONFIG] Target DOCX: D:\Code\AI\rag-mastery\data\docx\proposal.docx


In [2]:
# Step 2: Standard Ingestion with Docx2txtLoader
loader = Docx2txtLoader(str(docx_path))
docx_docs = loader.load()

print(f"[INFO] Docx2txtLoader extracted {len(docx_docs)} document(s).")
print(f"Content Preview ({len(docx_docs[0].page_content)} chars):")
print(docx_docs[0].page_content[:200] + "...")

# Assertions
assert len(docx_docs) == 1, "Docx2txtLoader must produce exactly one aggregated Document."
assert len(docx_docs[0].page_content) > 0, "Document content must not be empty."


[INFO] Docx2txtLoader extracted 1 document(s).
Content Preview (740 chars):
Project Proposal: Enterprise RAG Implementation

Executive Summary

This project aims to implement a state-of-the-art Retrieval-Augmented Generation (RAG) system for enterprise knowledge management. T...


In [3]:
# Step 3: Granular Extraction with python-docx (Headings & Tables)
from langchain_core.documents import Document

doc = docx.Document(str(docx_path))
structured_sections = []

# Extract paragraphs categorized by style
for p in doc.paragraphs:
    text = p.text.strip()
    if text:
        is_heading = p.style.name.startswith("Heading")
        structured_sections.append(
            Document(
                page_content=text,
                metadata={"source": docx_path.name, "is_heading": is_heading, "style": p.style.name}
            )
        )

# Extract tables as markdown formatted text
for tbl_idx, table in enumerate(doc.tables, start=1):
    rows_data = []
    for row in table.rows:
        row_text = [cell.text.strip().replace("\n", " ") for cell in row.cells]
        rows_data.append(" | ".join(row_text))
    table_content = "\n".join(rows_data)
    structured_sections.append(
        Document(
            page_content=table_content,
            metadata={"source": docx_path.name, "element_type": "table", "table_index": tbl_idx}
        )
    )

print(f"[INFO] Extracted {len(structured_sections)} structured elements (paragraphs + tables).")
# Assertions
assert len(structured_sections) > 0, "Granular extraction must yield document sections."
print("[SUCCESS] Word document parsing and tabular extraction verified.")


[INFO] Extracted 9 structured elements (paragraphs + tables).
[SUCCESS] Word document parsing and tabular extraction verified.
